[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/spectral_cloud_tools.ipynb)

# Spectral Library and Cloud Helpers

This notebook demonstrates spectral library matching and chunk planning for cloud-native hyperspectral workflows.

In [ ]:
# %pip install hypercoast

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import xarray as xr

import hypercoast

## Build and save a small spectral library

In [ ]:
wavelengths = np.array([450.0, 550.0, 650.0, 750.0, 850.0])
library = hypercoast.SpectralLibrary(
    wavelengths=wavelengths,
    spectra={
        "clear_water": np.array([0.015, 0.025, 0.010, 0.006, 0.004]),
        "turbid_water": np.array([0.030, 0.055, 0.045, 0.030, 0.020]),
        "vegetation": np.array([0.030, 0.060, 0.040, 0.350, 0.500]),
    },
)
library.to_dataframe()

In [ ]:
with TemporaryDirectory() as tmpdir:
    csv_path = Path(tmpdir) / "spectral-library.csv"
    hypercoast.write_spectral_library(library, csv_path)
    reloaded = hypercoast.read_spectral_library(csv_path)
    display(reloaded.to_dataframe())

## Match spectra

In [ ]:
observed = np.array(
    [
        [0.016, 0.024, 0.011, 0.006, 0.004],
        [0.029, 0.054, 0.046, 0.031, 0.019],
    ]
)

hypercoast.match_spectra(observed, library, method="sam", top_k=2)

In [ ]:
spectrum = observed[0]
continuum_removed = hypercoast.continuum_remove(spectrum, wavelengths)
derivative = hypercoast.derivative_spectrum(spectrum, wavelengths)

continuum_removed, derivative

## Plan cloud-native chunks

Use `suggest_chunks` before opening or exporting large hyperspectral scenes with Dask-backed xarray workflows.

In [ ]:
cube = xr.DataArray(
    np.zeros((128, 120, 160), dtype="float32"),
    dims=("wavelength", "y", "x"),
    coords={"wavelength": np.linspace(400, 900, 128)},
    name="reflectance",
)

hypercoast.suggest_chunks(cube, target_pixels=25_000)